# Yeni bir telefonla çekilen toprak fotoğrafları
Üç komşu modelini iki telefonla çekilmiş aynı numuneler üzerinde karşılaştırıyoruz.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.preprocessing import StandardScaler

root = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("work/soil-data")
data = next(root.rglob("Training_labels_updated.csv")).parent
labels = pd.read_csv(data / "Training_labels_updated.csv").set_index("sample_id")
cameras = pd.read_csv(data / "ppm_updated.csv")
photos = sorted(data.joinpath("Training-All_Photos_updated").rglob("*.jpg"))
print(f"{len(labels)} numune, {len(photos)} eğitim fotoğrafı")

24 numune, 127 eğitim fotoğrafı


## 1. Fotoğrafı bir avuç sayıyla anlatmak
Bütün fotoğrafların merkezinden yaklaşık 100 × 100 mm alan alıyoruz. Kamera tablosundaki piksel/mm değerini gerçek dosyanın çözünürlüğüne uyarlıyoruz. Bu kalibrasyon, çekim uzaklığının kamera tablosunda varsayılana yakın olduğunu kabul eder.

Bu alanı griye çevirip 400 × 400 piksele getiriyoruz. Yatay ve dikey yönde, birbirinden 0,25 mm ile 8 mm arasında uzakta olan noktaların ne kadar farklı aydınlıkta göründüğünü ölçüyoruz. Böylece her fotoğrafta aynı altı fiziksel uzaklığı karşılaştırıyoruz. Her numunenin fotoğraflarını, her ölçünün ortanca değerini alarak birleştiriyoruz.

In [2]:
def physical_patch(path, camera):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        w, h = image.size
        pixels_per_mm = camera.ppm * max(w, h) / max(camera.width, camera.height)
        side = round(100 * pixels_per_mm)
        assert 0 < side <= min(w, h)
        x, y = (w - side) // 2, (h - side) // 2
        return image.crop((x, y, x + side, y + side)).convert("RGB").resize((400, 400), Image.Resampling.LANCZOS)

def texture(patch):
    gray = np.asarray(patch.convert("L"), dtype=float)
    low, high = np.percentile(gray, [5, 95])
    gray = np.clip((gray - low) / max(high - low, 1.0), 0, 1)
    values = []
    for offset in [1, 2, 4, 8, 16, 32]:
        horizontal = np.abs(gray[:, offset:] - gray[:, :-offset]).mean()
        vertical = np.abs(gray[offset:, :] - gray[:-offset, :]).mean()
        values.append((horizontal + vertical) / 2)
    return values

camera_by_name = {c.phone.replace(" ", "_").lower(): c for c in cameras.itertuples()}
rows = []
examples = {}
for path in photos:
    phone, sample, number = path.stem.rsplit("_", 2)
    patch = physical_patch(path, camera_by_name[phone.lower()])
    rows.append([sample, *texture(patch)])
    examples.setdefault(sample, patch)
features = pd.DataFrame(rows, columns=["sample_id", "0.25", "0.5", "1", "2", "4", "8"])
features = features.groupby("sample_id").median()
assert set(features.index) == set(labels.index)
features = features.loc[labels.index]

## İki telefonla da çekilmiş numuneler

Motorola Edge ve Samsung A52 fotoğraflarını ayrı tutuyoruz. Her toprak ve telefon için altı doku ölçüsünün ortancasını alıyoruz. Her iki yönde aynı eğitim ve değerlendirme numunelerini kullanmak için yalnızca ortak numuneleri seçiyoruz.

In [3]:
phones = ["Motorola_Edge", "Samsung_A52"]
image_rows = pd.DataFrame(rows, columns=["numune", *features.columns])
image_rows["telefon"] = [path.stem.rsplit("_", 2)[0] for path in photos]
soil_phone = image_rows.groupby(["telefon", "numune"])[features.columns.tolist()].median()
by_phone = {p: soil_phone.loc[p] for p in phones}
paired = labels.index[labels.index.isin(by_phone[phones[0]].index) & labels.index.isin(by_phone[phones[1]].index)]
assert len(paired) == 21 and paired.is_unique
print(f"{len(paired)} ortak numune; her turda {len(paired)-1} eğitim numunesi.")

21 ortak numune; her turda 20 eğitim numunesi.


## Her numuneyi iki telefonla da sınama

Her turda aynı numunenin tüm fotoğraflarını eğitimden çıkarıyoruz. Seçilen eğitim telefonunda kalan 20 numuneyle ölçeklemeyi ve üç komşu modelini kuruyoruz. Aynı modelle saklanan numunenin iki telefonla çekilmiş görüntülerini ayrı ayrı değerlendiriyoruz. Hata, 11 boyuttaki tahmin eğrisinin laboratuvar eğrisinden logaritmik ağırlıklı uzaklığıdır (EMD). Düşük değer daha iyi.

In [4]:
widths = np.diff(np.log10(labels.columns.astype(float).to_numpy()))
records = []
for sample in paired:
    others = paired.drop(sample)
    for training_phone in phones:
        train = by_phone[training_phone].loc[others]
        assert sample not in train.index and len(train) == 20
        scaler = StandardScaler().fit(train)
        reference = scaler.transform(train)
        for viewing_phone in phones:
            held_out = by_phone[viewing_phone].loc[[sample]]
            distance = np.linalg.norm(reference - scaler.transform(held_out), axis=1)
            nearest = train.index[np.argsort(distance, kind="stable")[:3]]
            curve = labels.loc[nearest].mean()
            assert len(set(nearest)) == 3 and sample not in nearest
            error = np.abs((curve - labels.loc[sample]).to_numpy()[:-1]) @ widths
            records.append([sample, training_phone, viewing_phone, error, curve["2"]])

results = pd.DataFrame(records, columns=["Numune", "Eğitim", "Çekim", "EMD", "2 mm altı (%)"])
assert len(results) == 84 and not results.duplicated(["Numune", "Eğitim", "Çekim"]).any()
scores = results.pivot(index=["Numune", "Eğitim"], columns="Çekim", values="EMD")
for phone, other in [phones, phones[::-1]]:
    selected = scores.xs(phone, level="Eğitim")
    gap = selected[other] - selected[phone]
    print(phone, "eğitimi")
    print(f"Aynı telefon {selected[phone].mean():.2f} | Diğer telefon {selected[other].mean():.2f}")
    print(f"Ortalama fark {gap.mean():+.2f} | Ortanca fark {gap.median():+.2f}")
    print("Diğer telefonda hata artan / azalan / aynı:", (gap > 1e-9).sum(), (gap < -1e-9).sum(), np.isclose(gap, 0).sum())

print("H038 laboratuvar ölçümü (2 mm altı, %):", labels.loc["H038", "2"])
print(results[results.Numune == "H038"].round(2).to_string(index=False))
print(scores.round(2).to_string())

Motorola_Edge eğitimi
Aynı telefon 45.98 | Diğer telefon 49.10
Ortalama fark +3.11 | Ortanca fark +0.00
Diğer telefonda hata artan / azalan / aynı: 10 8 3
Samsung_A52 eğitimi
Aynı telefon 45.01 | Diğer telefon 45.26
Ortalama fark +0.26 | Ortanca fark -0.01
Diğer telefonda hata artan / azalan / aynı: 7 11 3
H038 laboratuvar ölçümü (2 mm altı, %): 47.2345
Numune        Eğitim         Çekim    EMD  2 mm altı (%)
  H038 Motorola_Edge Motorola_Edge 168.25          97.68
  H038 Motorola_Edge   Samsung_A52 129.21          96.24
  H038   Samsung_A52 Motorola_Edge 137.94          98.37
  H038   Samsung_A52   Samsung_A52 102.18          84.11
Çekim                 Motorola_Edge  Samsung_A52
Numune Eğitim                                   
F827   Motorola_Edge          50.31        62.26
       Samsung_A52            53.54        59.76
G190   Motorola_Edge          19.22        25.62
       Samsung_A52            23.61        27.19
H030   Motorola_Edge          30.27        30.27
       Samsung_A